In [ ]:
from pathlib import Path
import time

import polars as pl
import scipy as sp

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from dqdvs import dqdv_histogram

from config import DATA_PATH
ZENODO_DATA_PATH = Path(DATA_PATH) / Path("HALF_CELL_OCVS_ZENODO")

In [11]:
################################ LOAD DATA  ###################################
all_filepaths = ZENODO_DATA_PATH.rglob("*.parquet")


def generate_results_plot():
    
    fig = make_subplots(cols=2, shared_yaxes=True, horizontal_spacing=0.01, column_widths=[0.7, 0.3])
    fig.update_xaxes(title_text="Cumulative Capacity / Ah", row=1, col=1)
    fig.update_xaxes(title_text="dq/dV / Ah V-1", row=1, col=2)
    fig.update_yaxes(title_text="Voltage / V", row=1, col=1)

    fig.update_layout(
        template="plotly_white",
        width=700, 
        height=400,
        legend=dict(
            orientation="h",    # horizontal
            yanchor="bottom",   # anchor legend's bottom
            y=1.02,             # place above the plot area
            xanchor="center",
            x=0.5
        ),)
    return fig

colors = px.colors.qualitative.Plotly

In [12]:
[filepath.name for filepath in all_filepaths if  ("20250514" in filepath.stem)]

['sintef__sintef-graphite-R2032-intelligent-063b77__20250514__gitt__RT.bdf.parquet',
 'sintef__sintef-graphite-R2032-intelligent-4ccc47__20250514__p-ocv__RT.bdf.parquet',
 'sintef__sintef-graphite-R2032-intelligent-677295__20250514__p-ocvhold__RT.bdf.parquet',
 'sintef__sintef-silicongraphite-R2032-intelligent1-175ef2__20250514__gitt__RT.bdf.parquet',
 'sintef__sintef-silicongraphite-R2032-intelligent1-70e745__20250514__p-ocv__RT.bdf.parquet',
 'sintef__sintef-silicongraphite-R2032-intelligent1-bdb7bc__20250514__gitthold__RT.bdf.parquet',
 'sintef__sintef-silicongraphite-R2032-intelligent1-e43ac0__20250514__p-ocvhold__RT.bdf.parquet',
 'sintef__sintef-silicongraphite-R2032-intelligent2-655454__20250514__p-ocvhold__RT.bdf.parquet',
 'sintef__sintef-silicongraphite-R2032-intelligent2-fd2719__20250514__p-ocv__RT.bdf.parquet']

## Graphite with histogram method

In [13]:
BIN_SIZE = 1e-3

df = pl.read_parquet(ZENODO_DATA_PATH / Path('sintef__sintef-graphite-R2032-intelligent-677295__20250514__p-ocvhold__RT.bdf.parquet'))

df_subset = (df
             .filter(pl.col('Step Index / 1')==2)
             .filter(pl.col('Cycle Count / 1')==3)
             )

q = df_subset['Cumulative Capacity / Ah'].to_numpy()
v = df_subset['Voltage / V'].to_numpy()

v_dqdv, dqdv = dqdv_histogram(q=q, v=v, bin_size=BIN_SIZE)

q_reconstructed = sp.integrate.cumulative_trapezoid(dqdv, v_dqdv, initial=0.0) + q[0]

fig = generate_results_plot()


fig.add_trace( go.Scatter(x=q, 
                          y=v, 
                          name="OCV", 
                          line=dict(color="Black", width=4)), 
                        row=1, col=1 )
fig.add_trace( go.Scatter(x=q_reconstructed, 
                          y=v_dqdv, 
                          name="OCV reconstructed", 
                          showlegend=False,
                          line=dict(color=colors[0], width=3, dash="solid")), 
                        row=1, col=1 )


fig.add_trace( go.Scatter(x=dqdv, 
                          y=v_dqdv, 
                          name="OCV histogram: 1 mV bin size", 
                          line=dict(color=colors[0], width=2)), 
                        row=1, col=2 )

fig.add_trace(go.Bar(x=dqdv, 
                     y=v_dqdv, 
                     name="Bins", 
                     orientation="h", 
                     opacity=0.5, 
                     marker=dict(color=colors[0]), 
                     showlegend=False), 
                    row=1, col=2 )

plateaus = [0.119, 0.1125, 0.0765]
for plateau in plateaus:
    fig.add_hline(y=plateau,  line_color=colors[2], line_width=1, line_dash = "dot", annotation_text=f"{plateau:.3f} V", annotation_position="top right", row=1, col=1)
    fig.add_hline(y=plateau,  line_color=colors[2], line_width=1, line_dash = "dot", row=1, col=2)



y_lims = [0.06, 0.13]
x1_lims = [347e-6, 1.99e-3]

fig.add_hline(y=y_lims[0],  line_color="Black", line_width=3)
fig.add_vline(x=x1_lims[0],  line_color="Black", line_width=3, row=1, col=1)
fig.add_vline(x=0,  line_color="Black", line_width=2, row=1, col=2)

fig.update_yaxes(range=y_lims, showgrid=False)
fig.update_xaxes(showgrid=False, range=x1_lims, row=1, col=1)
fig.update_xaxes(showgrid=False, row=1, col=2)
# fig.update_xaxes(range=[0, 3e-3], row=1, col=1)
fig.update_layout(title="Graphite vs Li: histogram")
fig.show()